In [6]:
# import boto3

# sts = boto3.client("sts")
# print(sts.get_caller_identity())

{'UserId': 'AROARCH7ZESHBYDBKINW3:SageMaker', 'Account': '073550865550', 'Arn': 'arn:aws:sts::073550865550:assumed-role/AmazonSageMakerServiceCatalogProductsUseRole/SageMaker', 'ResponseMetadata': {'RequestId': 'ed303277-a3e1-42e6-bcbf-4906d86117b6', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'ed303277-a3e1-42e6-bcbf-4906d86117b6', 'x-amz-sts-extended-request-id': 'MTphcC1zb3V0aC0xOlM6MTc3NDQ0NDE3MTUwOTpSOm9zTGZ2VWgy', 'content-type': 'text/xml', 'content-length': '469', 'date': 'Wed, 25 Mar 2026 13:09:31 GMT'}, 'RetryAttempts': 0}}


In [7]:
import boto3
session = boto3.Session()
s3 = session.client('s3', region_name='ap-south-1')

s3.download_file(
    'pred-maintain-model-rodix',
    'predictive_maintenance_model.pkl',
    'predictive_maintenance_model.pkl'
)

In [15]:
!pip install xgboost==1.7.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.3/200.3 MB 50.1 MB/s  0:00:04m0:00:0100:01


In [16]:
import joblib

model = joblib.load("predictive_maintenance_model.pkl")

[13:19:32] WARNING: ../src/learner.cc:553: 
  If you are loading a serialized model (like pickle in Python, RDS in R) generated by
  older XGBoost, please export the model by calling `Booster.save_model` from that version
  first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/latest/tutorials/saving_model.html

  for more details about differences between saving model and serializing.



In [18]:
import sagemaker
from sagemaker.sklearn.model import SKLearnModel

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [ ]:
role = sagemaker.get_execution_role()

In [44]:
import boto3

s3 = boto3.client("s3")

s3.upload_file("model.tar.gz", "pred-maintain-model-rodix", "model/model.tar.gz")

In [45]:
from sagemaker.xgboost.model import XGBoostModel
model = XGBoostModel(
    model_data="s3://pred-maintain-model-rodix/model/model.tar.gz",
    role=role,
    entry_point="inference.py",
    framework_version="1.7-1"
)

In [46]:
predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large"
)

------!

In [47]:
predictor

In [55]:
# Updating endpoint
predictor.update_endpoint(
    initial_instance_count=1,
    instance_type="ml.m5.large"
)

------!

In [56]:
import boto3

runtime = boto3.client("sagemaker-runtime", region_name="ap-south-1")

# Real sample from my dataset - 10 features: device, metric1-metric9
sample = [[266, 215630672, 55, 6, 11, 13, 407438, 0, 0, 7]]

response = runtime.invoke_endpoint(
    EndpointName="sagemaker-xgboost-2026-03-26-06-29-01-426", 
    ContentType="text/plain",
    Body=str(sample)
)

prediction = response["Body"].read().decode("utf-8")
print("Prediction:", prediction)
print("0 = No failure, 1 = Failure")

Prediction: [0]
0 = No failure, 1 = Failure


In [53]:
import boto3

logs = boto3.client("logs", region_name="ap-south-1")

log_group = "/aws/sagemaker/Endpoints/sagemaker-xgboost-2026-03-26-06-29-01-426" 

streams = logs.describe_log_streams(
    logGroupName=log_group,
    orderBy="LastEventTime",
    descending=True
)

stream_name = streams["logStreams"][0]["logStreamName"]

events = logs.get_log_events(
    logGroupName=log_group,
    logStreamName=stream_name,
    limit=30
)

for e in events["events"]:
    print(e["message"])

169.254.178.2 - - [26/Mar/2026:06:44:03 +0000] "GET /ping HTTP/1.1" 200 0 "-" "AHC/2.0"
169.254.178.2 - - [26/Mar/2026:06:44:08 +0000] "GET /ping HTTP/1.1" 200 0 "-" "AHC/2.0"
169.254.178.2 - - [26/Mar/2026:06:44:13 +0000] "GET /ping HTTP/1.1" 200 0 "-" "AHC/2.0"
169.254.178.2 - - [26/Mar/2026:06:44:18 +0000] "GET /ping HTTP/1.1" 200 0 "-" "AHC/2.0"
169.254.178.2 - - [26/Mar/2026:06:44:23 +0000] "GET /ping HTTP/1.1" 200 0 "-" "AHC/2.0"
169.254.178.2 - - [26/Mar/2026:06:44:28 +0000] "GET /ping HTTP/1.1" 200 0 "-" "AHC/2.0"
169.254.178.2 - - [26/Mar/2026:06:44:33 +0000] "GET /ping HTTP/1.1" 200 0 "-" "AHC/2.0"
169.254.178.2 - - [26/Mar/2026:06:44:38 +0000] "GET /ping HTTP/1.1" 200 0 "-" "AHC/2.0"
169.254.178.2 - - [26/Mar/2026:06:44:43 +0000] "GET /ping HTTP/1.1" 200 0 "-" "AHC/2.0"
169.254.178.2 - - [26/Mar/2026:06:44:48 +0000] "GET /ping HTTP/1.1" 200 0 "-" "AHC/2.0"
169.254.178.2 - - [26/Mar/2026:06:44:53 +0000] "GET /ping HTTP/1.1" 200 0 "-" "AHC/2.0"
[2026-03-26:06:44:56:ERROR] Exce

In [54]:
import xgboost
print(xgboost.__version__)

1.7.6


In [57]:
!pip install requests

145018.28s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


In [58]:
import requests
import json

url = "https://su40h5dbj7.execute-api.ap-south-1.amazonaws.com/prod/predict"

# Test normal case
payload = {"input": [[266, 215630672, 55, 6, 11, 13, 407438, 0, 0, 7]]}

response = requests.post(url, json=payload)
print("Status:", response.status_code)
print("Response:", response.json())

Status: 200
Response: {'prediction': 0, 'alert_sent': False}


In [59]:
# Test failure case - should trigger email
payload = {"input": [[266, 215630672, 55, 6, 11, 13, 407438, 1, 1, 100]]}

response = requests.post(url, json=payload)
print("Status:", response.status_code)
print("Response:", response.json())

Status: 200
Response: {'prediction': 1, 'alert_sent': True}
